In [0]:
df_silver_st = spark.read.table("hdfc_ergo.hdfc_silver.dim_service_types")

In [0]:
from pyspark.sql.functions import row_number, col
from pyspark.sql.window import Window

# 1. Define window specification ordered by the business key
window_spec = Window.orderBy("service_type_id")

# 2. Generate Surrogate Key: AUTO_INCREMENT
df_gold_st = df_silver_st.withColumn("service_type_sk", row_number().over(window_spec))

# 3. Select ONLY the columns required in Gold (Column Pruning & Reordering)
df_gold_st = df_gold_st.select(
    "service_type_sk",
    "service_type_id",       # Business Key
    "service_type_name",     # Attribute
    "service_category",      # Hierarchy Attribute
    "requires_prior_auth",   # Attribute (Boolean)
    "reference_price"        # Measure/Attribute (Decimal)
)

display(df_gold_st.limit(5))

In [0]:
gold_table_fqn = "hdfc_ergo.hdfc_gold.dim_service_types"

df_gold_st.write.mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable(gold_table_fqn)

print(f"✅ Successfully wrote Dim_Service_Types to Gold layer.")